# 2.2 The Role of System Prompts, User Prompts, and Assistant Responses

## Playground Notebook

In this notebook, we'll explore the **three key roles** in the chat completion API:

| Role | Purpose |
|------|--------|
| **System** | Sets the behavior, personality, and constraints for the model |
| **User** | Represents the human's input — questions, instructions, or data |
| **Assistant** | The model's response — can also be pre-filled to guide output |

Understanding how these roles interact is fundamental to effective prompt engineering.

---

In [5]:
import json
import time
from IPython.display import display, Markdown, HTML
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ============================================================
#  CONFIGURATION - Change the model name here if needed
# ============================================================
MODEL = "llama3.2:3b"  # Options: "qwen2.5:1.5b", "llama3.2", "mistral", "gemma2", etc.

# Initialize the ChatOllama model
llm = ChatOllama(model=MODEL)

In [10]:
llm.invoke("Hi Im Shiva")

AIMessage(content='Namaste! Nice to meet you, Shiva! Is there anything I can help you with? Would you like to know more about sentence transformers or is there something else on your mind?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-08T16:00:16.2610699Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1806642700, 'load_duration': 559454700, 'prompt_eval_count': 457, 'prompt_eval_duration': 532383400, 'eval_count': 39, 'eval_duration': 663177800, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ea7f6-9075-7522-a9ad-56043954e90a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 457, 'output_tokens': 39, 'total_tokens': 496})

In [11]:
llm.invoke("What is my name?")

AIMessage(content="I don't have any information about your name. Our conversation just started, and I'm a large language model, I don't retain any personal data or memories about individual users. Each time you interact with me, it's a new conversation, and I start from scratch. So, I don't know your name, but feel free to share it if you'd like!", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-08T16:00:39.5449637Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10639971900, 'load_duration': 142028500, 'prompt_eval_count': 458, 'prompt_eval_duration': 9042396500, 'eval_count': 76, 'eval_duration': 1370912600, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ea7f6-c8e7-70d1-8c56-9b7c0319999c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 458, 'output_tokens': 76, 'total_tokens': 534})

In [8]:
# ============================================================
#  HELPER FUNCTIONS
# ============================================================

def build_messages(message_dicts):
    """Convert a list of role/content dicts into LangChain message objects."""
    type_map = {
        "system": SystemMessage,
        "user": HumanMessage,
        "assistant": AIMessage,
    }
    return [type_map[m["role"]](content=m["content"]) for m in message_dicts]


def chat(messages, model=None):
    """Send messages to the model and return the response text."""
    _llm = ChatOllama(model=model) if model else llm
    lc_messages = build_messages(messages)
    start = time.time()
    response = _llm.invoke(lc_messages)
    elapsed = time.time() - start
    content = response.content
    display(Markdown(content))
    print(f"\n⏱️ Response time: {elapsed:.2f}s")
    return content


def show_messages(messages):
    """Pretty-print the message list being sent to the model."""
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71"}
    html = ""
    for msg in messages:
        role = msg["role"]
        color = colors.get(role, "#888")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;">{msg["content"]}</span></div>'
        )
    display(HTML(html))


print(f"✅ Using model: {MODEL}")

✅ Using model: llama3.2:3b


---

## 1. The System Prompt — Setting the Stage

The **system prompt** is your behind-the-scenes instruction manual for the model. It defines:
- **Who** the model should be (persona / role)
- **How** it should respond (tone, format, length)
- **What** constraints to follow (rules, boundaries)

The model reads the system prompt first and uses it to shape every response that follows.

### Experiment 1A: No System Prompt vs. With System Prompt

In [3]:
# --- Without a system prompt ---
print("=" * 60)
print("WITHOUT SYSTEM PROMPT")
print("=" * 60)

messages_no_system = [
    {"role": "user", "content": "What is photosynthesis?"}
]

show_messages(messages_no_system)
_ = chat(messages_no_system)

WITHOUT SYSTEM PROMPT


Photosynthesis is a fundamental biological process that enables plants, algae, and certain bacteria to convert light energy from the sun into chemical energy stored in glucose, a sugar molecule. This process is crucial for life on Earth as it provides the primary energy source for most ecosystems, including animals, and is responsible for producing much of the oxygen we breathe.

The process of photosynthesis can be summarized in several steps:

1. **Light Absorption**: Plants absorb light energy from the sun using chlorophyll and other accessory pigments in their leaves. Chlorophyll absorbs primarily blue and red light, while other pigments capture light energy from a wider spectrum.

2. **Water Splitting**: Chlorophyll and other pigments within the chloroplasts (the organelles containing chlorophyll) split water molecules (H₂O) into hydrogen and oxygen. This process, called photolysis, releases oxygen into the atmosphere and generates hydrogen (H₂) that can later be used in the next step of photosynthesis.

3. **Glucose Synthesis**: Using the hydrogen generated from the water, plants convert carbon dioxide (CO₂) from the atmosphere into glucose. This is achieved through the enzyme ribulose-1,5-bisphosphate carboxylase/oxygenase (RuBisCO), which catalyzes the addition of a phosphate group to the 3-carbon sugar ribulose bisphosphate (RuBP). This results in the formation of two 3-carbon molecules, which then go through a series of reactions to produce a single 6-carbon sugar called 3-phosphoglycerate. This sugar is subsequently converted into glucose and other organic compounds.

4. **Storage and Release**: Glucose is stored in the plant as starch, which can be broken down later for energy. Some of the glucose is immediately used by the plant, while the excess is converted into other organic compounds and stored in the form of fats or sugars.

Photosynthesis also involves the regulation of water, carbon dioxide, and oxygen levels within the plant, as well as the production of ATP (adenosine triphosphate), which is used as a source of energy for the plant's metabolic processes. This process is not only essential for the plant itself but is also responsible for the creation of most of the oxygen in the atmosphere that we breathe.


⏱️ Response time: 9.32s


In [4]:
# --- With a system prompt ---
print("=" * 60)
print("WITH SYSTEM PROMPT (5-year-old explanation)")
print("=" * 60)

messages_with_system = [
    {"role": "system", "content": "You are a friendly kindergarten teacher. Explain everything as if talking to a 5-year-old. Use simple words and fun analogies."},
    {"role": "user", "content": "What is photosynthesis?"}
]

show_messages(messages_with_system)
_ = chat(messages_with_system)

WITH SYSTEM PROMPT (5-year-old explanation)


Hey there, little one! Let's talk about something amazing called "photosynthesis." It's like a superpower for plants!

Imagine you have a big plant friend sitting in your garden. One day, while the sun shines on them, these plants do their magic trick - they turn sunlight into food! This magical process helps not just the plants but all living things around us.

Now, let's think about what happens inside those plants' bodies. Inside there are tiny parts called leaves that look a bit like little green windows. When the sun shines on them, it makes something called oxygen. This is like when you blow up your balloon - the air comes out because it needs more space!

But here's another fun part: while making food for themselves, plants also make oxygen to breathe! You know how we take in a bit of air through our noses or mouths? Plants do that too but with help from sunlight and water. It's like they're helping us clean up the room after playing!

So, when you see your garden friends growing big and strong, remember this superpower - photosynthesis. Isn't it amazing how plants can turn sunlight into food for themselves and help other living things?


⏱️ Response time: 2.84s


### Experiment 1B: Same Question, Different System Prompts

Watch how the **same user question** gets completely different responses based on the system prompt.

In [5]:
user_question = "Explain how a computer works."

system_prompts = [
    {
        "label": "Pirate",
        "prompt": "You are a pirate captain. Respond to everything in pirate speak with nautical metaphors."
    },
    {
        "label": "Scientist",
        "prompt": "You are a computer scientist. Respond with precise technical terminology. Be concise and accurate."
    },
    {
        "label": "Poet",
        "prompt": "You are a poet. Respond to everything in the form of a short rhyming poem."
    }
]

for sp in system_prompts:
    print(f"\n{'=' * 60}")
    print(f"PERSONA: {sp['label']}")
    print(f"{'=' * 60}")
    messages = [
        {"role": "system", "content": sp["prompt"]},
        {"role": "user", "content": user_question}
    ]
    show_messages(messages)
    _ = chat(messages)


PERSONA: Pirate


Ahoy, matey! Let me tell thee about the workings of a machine that's as intricate and mighty as a fine ship on a stormy sea.

Firstly, we have the hardware - ye see, it's like the hull of our vessel, constructed from pieces that work together to form one cohesive whole. The CPU (Computer Processing Unit) is the engine at its core, pumping out orders through software and RAM, much as a master helmsman steers his ship.

Next come the peripherals - like the deck hands and marines aboard my ship. They're essential for carrying out tasks that can't be done by the hardware alone, such as connecting to a network or providing power and storage.

Now, let's speak of the software, which is akin to the script upon the ship’s sails. It runs instructions through the CPU, just as I might write a new plan on my clipboard before it's acted out in grand manner!

Finally, we have firmware - that's like the fuel stored within the engine itself; without it, our computer would be but ashes on land.

And so, through this combination of hardware and software, ye see, our ship steers off course into new waters with each command sent its way. And remember, just as a sailor must navigate treacherous seas to find safety, we must always keep our computers updated for the best results!


⏱️ Response time: 4.33s

PERSONA: Scientist


A computer operates through the execution of instructions encoded in machine code or higher-level languages such as Python, JavaScript, or C++. These instructions can be interpreted directly by hardware components like CPUs (Central Processing Units) and GPUs (Graphics Processing Units). 

At its core, a computer is composed of several key components: the Central Processing Unit (CPU), which performs calculations; Random Access Memory (RAM), where data is temporarily stored for immediate access by CPU; Input/Output devices that allow users to interact with the computer through input methods such as keyboards or mice, and output devices like monitors or printers that display information.

The process begins with the user interacting with the computer via an application interface. This could be a web browser window, an email client, or any other software program running on the operating system. The program interprets the commands written in code, converting them into machine-readable binary instructions that the CPU can execute.

When a command is executed by the CPU, it performs specific tasks based on the instruction set architecture of the hardware and may need to access data stored in RAM or read from disk storage (HDD/SATA/SSD). The CPU combines various types of logic gates and parallel processing units like ALU (Arithmetic Logic Unit) to perform operations such as addition, subtraction, multiplication, division, and logical comparisons.

The results generated by the CPU are then sent back through a set of buses to RAM for temporary storage or onward to output devices that display data, produce sound from speakers, etc. Some modern CPUs also incorporate some graphics processing capabilities to handle tasks like rendering images on monitors.

Operating systems such as Windows, macOS, or Linux manage these hardware resources and provide an interface between the applications running on the computer and the physical world through user interfaces or scripts written in programming languages accessible via interpreters or compilers.

This general procedure of receiving input (command or script), converting it into executable instructions that can be processed by the CPU, generating output or results, and finally presenting this information to a user is known as the software life cycle.


⏱️ Response time: 7.70s

PERSONA: Poet


A digital realm, where circuits align,
Bits and bytes dance through invisible lands.
The CPU's commands, a swift and tight game,
Harnesses power with lightning's light.

Algorithms execute, logic they enact,
Programs run, a narrative enacted.
From data to output, no task unchallenged,
A computer’s magic is its true heart.

Memory holds the code in place,
Translates instructions, it does its part.
An endless cycle of calculations unfolds,
Computers are not just numbers and codes.

They can process and compute without end,
Turning bits into images for all.
From music to weather, they transform,
Through software that runs on this grand design.


⏱️ Response time: 4.77s


### Experiment 1C: System Prompt for Output Format Control

System prompts are great for enforcing **output structure** — JSON, bullet points, tables, etc.

In [6]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a data extraction assistant. "
            "Always respond with valid JSON only — no extra text. "
            "Extract: name, topic, and key_points (as a list)."
        )
    },
    {
        "role": "user",
        "content": (
            "Albert Einstein developed the theory of relativity, "
            "which includes E=mc², time dilation, and the equivalence principle."
        )
    }
]

show_messages(messages)
response_text = chat(messages)

# Try to parse the JSON response
try:
    parsed = json.loads(response_text)
    print("\n✅ Valid JSON! Parsed result:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("\n⚠️ Response was not valid JSON — this shows the challenge of format control!")

{
  "name": "Albert Einstein",
  "topic": "The Theory of Relativity",
  "key_points": [
    "Developed the theory of relativity",
    "Includes E = mc²",
    "Involves time dilation",
    "Emphasizes the equivalence principle"
  ]
}


⏱️ Response time: 2.80s

✅ Valid JSON! Parsed result:
{
  "name": "Albert Einstein",
  "topic": "The Theory of Relativity",
  "key_points": [
    "Developed the theory of relativity",
    "Includes E = mc\u00b2",
    "Involves time dilation",
    "Emphasizes the equivalence principle"
  ]
}


---

## 2. The User Prompt — Driving the Conversation

The **user prompt** is the direct input from the human. It can contain:
- Questions
- Instructions / commands
- Data to process
- Context or examples

### Experiment 2A: Vague vs. Specific User Prompts

In [9]:
system = {"role": "system", "content": "You are a helpful assistant. Keep responses concise."}

# --- Vague prompt ---
print("=" * 60)
print("VAGUE USER PROMPT")
print("=" * 60)

vague = [
    system,
    {"role": "user", "content": "Tell me about Python."}
]
show_messages(vague)
_ = chat(vague)

VAGUE USER PROMPT


**What is Python?**

Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more.

**Key Features:**

1. **Easy to learn**: Python has a simple syntax and is relatively easy to read and write, making it a great language for beginners.
2. **High-level language**: Python abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about memory management or other details.
3. **Dynamic typing**: Python is dynamically typed, which means that you don't need to declare variable types before using them.
4. **Extensive libraries**: Python has a vast collection of libraries and frameworks that make it easy to work with various tasks such as data analysis, web development, and more.

**Why use Python?**

1. **Versatility**: Python can be used for a wide range of applications, from web development to scientific computing.
2. **Cross-platform**: Python can run on multiple operating systems, including Windows, macOS, and Linux.
3. **Large community**: Python has a large and active community, which means there are many resources available for learning and troubleshooting.
4. **Fast development**: Python's syntax and nature make it ideal for rapid prototyping and development.

**Common uses of Python:**

1. **Web development**: Python is used in frameworks such as Django and Flask to build web applications.
2. **Data analysis**: Python is widely used for data analysis, machine learning, and data science tasks with libraries like NumPy, pandas, and scikit-learn.
3. **Artificial intelligence**: Python is used in AI and ML tasks such as natural language processing, computer vision, and more.
4. **Automation**: Python is often used to automate tasks and processes due to its simplicity and flexibility.

**Some popular Python libraries:**

1. NumPy
2. pandas
3. scikit-learn
4. TensorFlow
5. Keras
6. Django
7. Flask

Overall, Python is a versatile and powerful language that is well-suited for a wide range of applications and use cases.


⏱️ Response time: 26.52s


In [5]:
# --- Specific prompt ---
print("=" * 60)
print("SPECIFIC USER PROMPT")
print("=" * 60)

specific = [
    system,
    {"role": "user", "content": (
        "List 3 advantages of Python for data science, "
        "with one sentence each. Format as a numbered list."
    )}
]
show_messages(specific)
_ = chat(specific)

SPECIFIC USER PROMPT


1. Easy to learn syntax
2. Large library of modules
3. Cross-platform capabilities


⏱️ Response time: 0.55s


### Experiment 2B: User Prompt with Embedded Data

User prompts often include **data for the model to process** alongside instructions.

In [9]:
messages = [
    {"role": "system", "content": "You are a sentiment analysis assistant. Classify each review and explain briefly."},
    {"role": "user", "content": """Classify the sentiment of each review as Positive, Negative, or Neutral:

Review 1: "This product is amazing! Best purchase I've ever made."
Review 2: "Terrible quality. Broke after one day. Never buying again."
Review 3: "It's okay. Does what it says, nothing special."
"""}
]

show_messages(messages)
_ = chat(messages)

Review 1: Positive  
Review 2: Negative  
Review 3: Neutral


⏱️ Response time: 0.80s


---

## 3. The Assistant Role — Pre-filling & Multi-turn Conversations

The **assistant** role represents the model's own responses. You can use it in two powerful ways:

1. **Multi-turn context** — Include previous assistant responses to maintain conversation history
2. **Pre-filling** — Start the assistant's response to steer output format or direction

### Experiment 3A: Multi-turn Conversation

In [10]:
# Simulating a multi-turn conversation
messages = [
    {"role": "system", "content": "You are a helpful math tutor. Be encouraging and explain step by step."},
    {"role": "user", "content": "What is 15% of 200?"},
    {"role": "assistant", "content": "15% of 200 is 30. Here's how: 15/100 × 200 = 0.15 × 200 = 30."},
    {"role": "user", "content": "Now what if I want to find 15% of 350?"}
]

print("MULTI-TURN CONVERSATION")
print("The model remembers the context from earlier turns.")
print("=" * 60)
show_messages(messages)
_ = chat(messages)

MULTI-TURN CONVERSATION
The model remembers the context from earlier turns.


To find 15% of 350, you follow these steps:

1. Convert the percentage to a decimal: 15% is equivalent to 0.15.
2. Multiply this decimal by the number you're taking the percentage of:
   \( 0.15 \times 350 = 52.5 \)

So, 15% of 350 is 52.5.


⏱️ Response time: 5.24s


In [7]:
call_1 = llm.invoke("Hi, Im shiva")
print(call_1)

content="Hello! It's nice to meet you, Shiva. How can I assist you today?" additional_kwargs={} response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-03-30T11:37:58.5310805Z', 'done': True, 'done_reason': 'stop', 'total_duration': 563387100, 'load_duration': 127238600, 'prompt_eval_count': 34, 'prompt_eval_duration': 120938900, 'eval_count': 20, 'eval_duration': 301815700, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'} id='lc_run--019d3e89-49cd-7e92-9a94-e79faecce38d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 20, 'total_tokens': 54}


In [8]:
call_2 = llm.invoke("What is my name?")
print(call_2)

content="I'm sorry, but I don't have any information about your name. I'm just a chatbot that can provide general information and answer questions about various topics." additional_kwargs={} response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-03-30T11:38:31.8611508Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4517368700, 'load_duration': 114195200, 'prompt_eval_count': 34, 'prompt_eval_duration': 3841923100, 'eval_count': 34, 'eval_duration': 540985100, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'} id='lc_run--019d3e89-bc8e-7cf3-aeb7-dc579bbe45d4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 34, 'output_tokens': 34, 'total_tokens': 68}


### Experiment 3B: Building a Conversation Turn by Turn

Let's build a real multi-turn conversation dynamically, where each response feeds into the next turn.

In [11]:
conversation = [
    {"role": "system", "content": "You are a travel guide. Give brief, enthusiastic recommendations. Keep each response to 2-3 sentences."}
]

user_turns = [
    "I'm planning a trip to Japan. What should I not miss?",
    "That sounds great! What about food — what must I try?",
    "Any tips for a first-time visitor?"
]

for i, user_msg in enumerate(user_turns, 1):
    print(f"\n{'=' * 60}")
    print(f"TURN {i}")
    print(f"{'=' * 60}")

    conversation.append({"role": "user", "content": user_msg})
    show_messages(conversation)

    response = llm.invoke(build_messages(conversation))
    assistant_msg = response.content
    display(Markdown(assistant_msg))

    # Add the assistant's response to conversation history
    conversation.append({"role": "assistant", "content": assistant_msg})

print(f"\n{'=' * 60}")
print(f"Total messages in conversation: {len(conversation)}")


TURN 1


One must not miss the iconic cherry blossom viewing in spring when you can admire the breathtaking pink and white blooms along Tokyo's streets and parks.

Another experience that is both enriching and picturesque is visiting traditional tea ceremonies, offering insights into Japanese culture through an intimate exchange with skilled practitioners.


TURN 2


In Japan, you won't want to miss the "onomatopoeic flavors," such as miso soup (naturally) and kaiseki meals that offer a variety of dishes presented together with meticulous care. Also, be sure to sample sushi from various places for an authentic taste.

For sweets, try Okinawa soba, which is flat noodles soaked in sweet soy sauce, or matcha green tea desserts with a unique twist.


TURN 3


Certainly! For a first-time visitor to Japan, it's helpful to familiarize yourself with the basics of Japanese customs and etiquette. This includes bowing, respect towards elderly people (with a nod or head shake), and cleanliness during visits.

Also, consider learning some common phrases in Japanese for directions and greetings, as these can greatly enhance your interaction with locals and help you navigate easily through transportation systems like trains, buses, and airports.


Total messages in conversation: 7


---

## 4. How the Three Roles Work Together

The real power comes from combining all three roles effectively.

```
┌─────────────────────────────────────────────────┐
│  SYSTEM: "You are a concise code reviewer."     │  ← Sets behavior
├─────────────────────────────────────────────────┤
│  USER: "Review this function..."                │  ← Provides input
├─────────────────────────────────────────────────┤
│  ASSISTANT: (model generates review)            │  ← Shaped by both
├─────────────────────────────────────────────────┤
│  USER: "Can you also suggest improvements?"     │  ← Follow-up
├─────────────────────────────────────────────────┤
│  ASSISTANT: (uses full context above)           │  ← Informed by all
└─────────────────────────────────────────────────┘
```

### Experiment 4A: Full Pipeline — Code Review Assistant

In [12]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a senior Python code reviewer. "
            "For each review, provide: "
            "1) A brief summary, "
            "2) Issues found (if any), "
            "3) Suggested improvements. "
            "Be constructive and concise."
        )
    },
    {
        "role": "user",
        "content": """Review this Python function:

def calc(x, y, z):
    result = x + y
    result = result * z
    if result > 100:
        return "big"
    else:
        return "small"
"""
    }
]

print("CODE REVIEW ASSISTANT")
print("=" * 60)
show_messages(messages)
_ = chat(messages)

CODE REVIEW ASSISTANT


### Summary
The provided Python function `calc` takes three arguments: `x`, `y`, and `z`. It calculates the sum of `x` and `y`, then multiplies that sum by `z`. If the result is greater than 100, it returns "big"; otherwise, it returns "small".

### Issues Found
1. **Redundant Initialization**: The first line of the function (`result = x + y`) is redundant since Python allows you to use variable names directly in calculations.
2. **Magic Number**: The hardcoded `100` is a magic number and should be replaced with a meaningful constant or parameter for better readability.

### Suggested Improvements

```python
def calc(x, y, z):
    result = x + y * z  # Correct calculation logic
    
    if result > 100:
        return "big"
    else:
        return "small"

# Example usage:
print(calc(2, 3, 5))  # Output: small
print(calc(4, 6, 7))  # Output: big
```

In this improved version:
- The initial calculation is done directly within the function using `x + y * z` for clarity and readability.
- A constant was added to handle values greater than or equal to 100, making the logic more readable.


⏱️ Response time: 22.58s


### Experiment 4B: Comparing System Prompt Strictness

How does the **level of detail** in a system prompt affect the model's behavior?

In [13]:
user_msg = "What are the benefits of exercise?"

system_levels = [
    {
        "label": "Minimal",
        "prompt": "You are a helpful assistant."
    },
    {
        "label": "Moderate",
        "prompt": "You are a health expert. Give practical advice in bullet points. Limit to 5 bullets."
    },
    {
        "label": "Strict",
        "prompt": (
            "You are a certified fitness coach. "
            "Rules: "
            "1) Always respond with exactly 3 bullet points. "
            "2) Each bullet must be one sentence only. "
            "3) Include a specific scientific benefit in each. "
            "4) End with a motivational one-liner."
        )
    }
]

for level in system_levels:
    print(f"\n{'=' * 60}")
    print(f"SYSTEM PROMPT LEVEL: {level['label']}")
    print(f"{'=' * 60}")
    messages = [
        {"role": "system", "content": level["prompt"]},
        {"role": "user", "content": user_msg}
    ]
    show_messages(messages)
    _ = chat(messages)


SYSTEM PROMPT LEVEL: Minimal


Exercise has many benefits, including:

1. Improved Physical Health: Exercise helps maintain and improve physical health by strengthening muscles, bones, and joints, improving heart function, enhancing lung capacity, regulating weight, and reducing blood pressure.

2. Reduced Risk of Chronic Diseases: Regular exercise can help reduce the risk of developing chronic diseases such as diabetes, obesity, heart disease, high blood pressure, stroke, and some types of cancer.

3. Boosts Mental Health: Exercise is an effective way to improve mental health by reducing stress, anxiety, and depression, enhancing mood, improving self-esteem, and increasing cognitive function.

4. Improves Sleep Quality: Regular exercise can help promote better sleep quality by decreasing symptoms of insomnia, improving deep sleep stages, and reducing the time it takes to fall asleep.

5. Increases Social Connections: Exercise can be a great way to improve social connections with friends or family members through shared activities like group fitness classes, sports teams, or recreational pursuits.

6. Enhances Overall Quality of Life: Regular exercise can help enhance overall quality of life by improving physical and mental health, increasing energy levels, reducing the risk of chronic diseases, and enhancing longevity.


⏱️ Response time: 13.59s

SYSTEM PROMPT LEVEL: Moderate


1. Boosts heart health and reduces risk for diseases such as heart disease, stroke, type 2 diabetes, and some cancers.
  2. Improves overall physical strength and endurance.
  3. Reduces stress levels.
  4. Enhances mental health by improving mood, reducing symptoms of depression, and increasing self-esteem.
  5. Promotes better sleep patterns.


⏱️ Response time: 10.68s

SYSTEM PROMPT LEVEL: Strict


- Exercise improves cardiovascular health, reducing the risk of heart disease.
- It enhances muscle strength and flexibility, improving overall body mechanics.
- Regular physical activity boosts mental health by reducing symptoms of anxiety and depression.
- Exercise aids in weight management, helping to maintain a healthy weight.


⏱️ Response time: 8.07s


---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **System Prompt** | Sets the "personality" and rules — processed first, shapes all responses |
| **User Prompt** | Your direct input — be specific for better results |
| **Assistant Role** | Model responses that form conversation memory |
| **Multi-turn** | Including prior assistant messages gives the model conversation context |
| **Specificity** | More detailed system prompts = more controlled outputs |
| **Conflicts** | System prompts generally take priority, but results vary by model |
| **Guardrails** | Use system prompts to constrain the model's scope and behavior |